# VLM From Scratch

### Part 1: Setup

In [ ]:
# Package Imports
!uv pip install -q transformers datasets torch torchvision pillow accelerate einops timm

In [ ]:
# Configs 
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    ViTModel,
    ViTImageProcessor,
)
from datasets import load_dataset
from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Print torch version
print(f"Torch version :{torch.__version__}")

In [ ]:
# Image format
%config InlineBackend.figure_format = 'retina'

### Vision Encoder (Vit)

the basic job of vit is to convert the images into a sequence of meaningful vectors.

Example: For a 224x224 image with 16x16 patches, we get: - (224/16) x (224/16) = 14 x 14 = 196 patches - Plus 1 CLS token = 197 tokens total - Each token has dimension 1024 (for ViT-Large)

In [ ]:
# Load pretrained ViT-Large (larger than ViT-Base for better performance)
vision_model_name = "google/vit-large-patch16-224"
vision_encoder = ViTModel.from_pretrained(vision_model_name)
image_processor = ViTImageProcessor.from_pretrained(vision_model_name)

# Freeze the vision encoder - we won't train it
for param in vision_encoder.parameters():
    param.requires_grad = False

vision_encoder = vision_encoder.to(device)
vision_encoder.eval()

print(f"Vision encoder hidden size: {vision_encoder.config.hidden_size}")
print(f"Number of patches: {(224//16)**2} + 1 CLS token = {(224//16)**2 + 1} tokens")

In [ ]:
# Test the vision encoder with a sample image
url = "https://image.api.playstation.com/vulcan/ap/rnd/202603/2506/93f66b17d66159f2a06f2f001b0e28cb485b524c9204797b.png"
response = requests.get(url)
sample_image = Image.open(BytesIO(response.content))

plt.figure(figsize=(6, 6))
plt.imshow(sample_image)
plt.title("Sample Image from COCO")
plt.axis('off')
plt.show()

# Process the image
inputs = image_processor(sample_image, return_tensors="pt").to(device)

with torch.no_grad():
    vision_outputs = vision_encoder(**inputs)

# Get the sequence of patch features (excluding pooler output)
image_features = vision_outputs.last_hidden_state
print(f"Image features shape: {image_features.shape}")
print(f"  - Batch size: {image_features.shape[0]}")
print(f"  - Sequence length (patches + CLS): {image_features.shape[1]}")
print(f"  - Hidden dimension: {image_features.shape[2]}")